## Clustering and Similarity Analysis

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from IPython.display import display

### K-Means Clustering

In [ ]:
# Load engineered member profiles
df = pd.read_csv(
    '/Users/meecee/Desktop/Github/Networking Recommendation System/data/processed/engineered_member_profiles.csv')

# Drop non-feature metadata columns
metadata_cols = ['email', 'full_name', 'job_title', 'organization',
                 'clean_sector', 'seniority_level_name', 'event_attendance_count'
]
X = df.drop(columns=metadata_cols)

print(f"Feature matrix shape for clustering: {X.shape}")

# Evaluate k from 2 to 8 using inertia and silhouette score (sample for silhouette if large)
inertias = []
silhouette_scores = []
db_scores = []
k_range = range(2,
9)

In [ ]:
# To evaluate silhouette on 7469 points efficiently, we can compute directly or use a random sample
sample_idx = np.random.RandomState(24).choice(len(X), size=2500, replace=False)
X_sample = X.iloc[sample_idx
]

for k in k_range:
    km = KMeans(n_clusters=k, random_state=24, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_sample, labels[sample_idx
])
    db = davies_bouldin_score(X, labels)
    silhouette_scores.append(sil)
    db_scores.append(db)
    print(
        f"k={k}: Inertia={km.inertia_:.2f}, Silhouette={sil:.4f}, DB-Index={db:.4f}")

In [ ]:
# Plot Elbow and Silhouette curves
fig, ax1 = plt.subplots(figsize=(8, 4))

color = 'tab:blue'
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (Elbow)', color=color)
ax1.plot(k_range, inertias, marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Silhouette Score', color=color)
ax2.plot(k_range, silhouette_scores, marker='s',
         color=color, linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Cluster Selection: Inertia vs Silhouette Score')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/Users/meecee/Desktop/Github/Networking Recommendation System/assets/clustering_evaluation.png', dpi=300)
plt.show()
plt.close()

In [ ]:
# Fit K-Means with k=5 for clean business interpretation (or k=4/k=6)
# Let's inspect cluster profiles with k=5
optimal_k = 5
km = KMeans(n_clusters=optimal_k, random_state=24, n_init=10)
df['cluster'] = km.fit_predict(X)

# Cluster breakdown by key traits
summary = []
for c in range(optimal_k):
    sub = df[df['cluster'] == c]
    top_sec = sub['clean_sector'].value_counts().head(2).to_dict()
    top_sen = sub['seniority_level_name'].value_counts().head(2).to_dict()
    avg_att = sub['event_attendance_count'].mean()
    med_att = sub['event_attendance_count']
    size = len(sub)
    summary.append({
        'Cluster': c,
        'Member Count': size,
        'Pct': f"{(size / len(df) * 100):.1f}%",
        'Avg Attendance': round(avg_att, 2),
        'Dominant Seniority': top_sen,
        'Dominant Sector': top_sec
})

summary_df = pd.DataFrame(summary)
print("\nCluster Summary:")
print(summary_df.to_string())

In [ ]:
# Dimensionality Reduction with PCA (2D) for visual inspection
pca = PCA(n_components=2, random_state=24)
coords = pca.fit_transform(X)
df['pca_x'] = coords[:, 0]
df['pca_y'] = coords[:, 1]

plt.figure(figsize=(9, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for c in range(optimal_k):
    sub = df[df['cluster'] == c]
    plt.scatter(sub['pca_x'], sub['pca_y'], label=f'Cluster {c}', alpha=0.5, s=20)

plt.title('Member Ecosystem Segmentation (PCA 2D Projection)')
plt.xlabel(f'PCA 1 ({pca.explained_variance_ratio_[0] * 100:.1f}% Variance)')
plt.ylabel(f'PCA 2 ({pca.explained_variance_ratio_[1] * 100:.1f}% Variance)')
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig('/Users/meecee/Desktop/Github/Networking Recommendation System/assets/cluster_pca_projection.png', dpi=300)
plt.show()
plt.close()

In [ ]:
# Save clustered dataset
df.to_csv('/Users/meecee/Desktop/Github/Networking Recommendation System/data/processed/clustered_member_profiles.csv', index=False)

In [ ]:
# Detailed breakdown of cluster personas
persona_names = {
    0: "Peripheral & Cold-Start Members",
    1: "Sector-Affiliated Observers",
    2: "Ecosystem Leaders & Founders",
    3: "Established Cyber & Cross-Sector Specialists",
    4: "Emerging Technical & Mid-Tier Practitioners"
}

for c, name in persona_names.items():
    sub = df[df['cluster'] == c]
    top_jobs = sub['job_title'].dropna().value_counts().head(3).to_dict()
    top_orgs = sub['organization'].dropna().value_counts().head(3).to_dict()
    print(f"Cluster {c}: {name}")
    print(
        f"Size: {len(sub)} ({len(sub)/len(df)*100:.1f}%) | Avg Events: {sub['event_attendance_count'].mean():.2f}")
    print(f"Top Titles: {top_jobs}")
    print(f"Top Orgs: {top_orgs}\n")

### Comparative Evaluation of 4 Clustering Paradigms

In [ ]:
# Load data
df_comp = pd.read_csv(
    '/Users/meecee/Desktop/Github/Networking Recommendation System/data/processed/engineered_member_profiles.csv')

metadata_cols = ['email', 'full_name', 'job_title', 'organization', 'clean_sector',
                 'seniority_level_name', 'event_attendance_count', 'cluster', 'pca_x', 'pca_y'
]
X_comp = df_comp.drop(columns=[c for c in metadata_cols if c in df_comp.columns])

In [ ]:
# We use a stratified or fixed representative sample of 2500 points for heavy metrics computation
np.random.seed(42)
sample_idx_comp = np.random.choice(len(X_comp), size=2500, replace=False)
X_sample_comp = X_comp.iloc[sample_idx_comp
]

# 1. K-Means (k=5 baseline from earlier)
kmeans = KMeans(n_clusters=5, random_state=24, n_init=10)
labels_km = kmeans.fit_predict(X_comp)

# 2. Agglomerative (Hierarchical) Clustering (k=5, Ward linkage)
# Note: For 7469 points, Agglomerative with ward linkage works fine in sklearn
agg = AgglomerativeClustering(n_clusters=5, metric='euclidean', linkage='ward')
labels_agg = agg.fit_predict(X_comp)

# 3. Gaussian Mixture Models (GMM) - Soft / Probabilistic clustering (k=5)
gmm = GaussianMixture(n_components=5, random_state=42)
labels_gmm = gmm.fit_predict(X_comp)

# 4. DBSCAN (Density-Based) - to test for arbitrary shapes and outliers / noise
# Let's test an eps that gives reasonable clusters in this normalized space
dbscan = DBSCAN(eps=0.5, min_samples=15)
labels_db = dbscan.fit_predict(X_comp)

In [ ]:
# Evaluate and summarize
models = {
    'K-Means (Centroid)': labels_km,
    'Agglomerative (Hierarchical)': labels_agg,
    'Gaussian Mixture (GMM - Probabilistic)': labels_gmm,
    'DBSCAN (Density-Based)': labels_db
}

eval_records = []

for name, labels in models.items():
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_count = np.sum(labels == -1) if -1 in labels else 0
    noise_pct = (noise_count / len(X_comp)) * 100

    # Calculate metrics on valid clusters (ignoring noise if DBSCAN, or on sample)
    if n_clusters > 1:
        # Sample labels
        sample_lbls = labels[sample_idx_comp]
        # Filter noise for silhouette if DBSCAN
        if name == 'DBSCAN (Density-Based)' and np.sum(sample_lbls != -1) > 10:
            valid_mask = sample_lbls != -1
            sil = silhouette_score(
                X_sample_comp[valid_mask], sample_lbls[valid_mask])
            valid_all = labels != -1
            db = davies_bouldin_score(X_comp[valid_all], labels[valid_all])
            ch = calinski_harabasz_score(X_comp[valid_all], labels[valid_all])
        else:
            sil = silhouette_score(X_sample_comp, sample_lbls)
            db = davies_bouldin_score(X_comp, labels)
            ch = calinski_harabasz_score(X_comp, labels)
    else:
        sil, db, ch = np.nan, np.nan, np.nan

    eval_records.append({
        'Algorithm': name,
        'Clusters Found': n_clusters,
        'Noise / Outlier %': f"{noise_pct:.1f}%",
        'Silhouette Score (↑)': round(sil,4),
        'Davies-Bouldin (↓)': round(db,4),
        'Calinski-Harabasz (↑)': round(ch,1)
})

eval_df = pd.DataFrame(eval_records)
print(eval_df.to_string(index=False))

In [ ]:
# Save comparative plot across PCA space for the four algorithms
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()
coords_x = df['pca_x']
coords_y = df['pca_y']

for idx, (name, lbls) in enumerate(models.items()):
    ax = axes[idx]
    if -1 in lbls:
        # Plot noise in gray
        noise_mask = (lbls == -1)
        ax.scatter(coords_x[noise_mask], coords_y[noise_mask],
                   c='lightgray', s=10, alpha=0.3, label='Noise/Outlier')
        scatter = ax.scatter(coords_x[~noise_mask], coords_y[~noise_mask], c=lbls[~noise_mask], cmap='tab10', s=15, alpha=0.6)
    else:
        scatter = ax.scatter(coords_x, coords_y, c=lbls, cmap='tab10', s=15, alpha=0.6)

    ax.set_title(
        f"{name}\nSil: {eval_records[idx]['Silhouette Score (↑)']} | DB: {eval_records[idx]['Davies-Bouldin (↓)']}", 
        fontsize=11, fontweight='bold')
    ax.set_xlabel('PCA 1')
    ax.set_ylabel('PCA 2')
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('/Users/meecee/Desktop/Github/Networking Recommendation System/assets/clustering_algorithm_comparison.png', dpi=300)
plt.show()
plt.close()
print("Saved clustering_algorithm_comparison.png")